# Loading EEG epochs with `tnsd_access`

This notebook demonstrates the epoch-loading workflow after the dataset has already been initialised. See `getting_started.ipynb` for dataset setup, metadata inspection, and on-demand download details.

Here we focus on actually fetching epochs into arrays:

1. Select a small trial table.
2. Load named channels and a cropped time window.
3. Average trials with `average_by`.
4. Iterate in batches for larger selections.

The important habit is to filter trials first and load only the channels/time window you need.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tnsd_access import TrialHandler

pd.set_option("display.max_columns", 40)

## 1. Point at an initialised dataset

`DATA_ROOT` should be the folder created in `getting_started.ipynb` with `init_dataset(...)`, or a local copy of the dataset.

In [ ]:
DATA_ROOT = Path("temporal-natural-scenes-dataset")
VERSION = "v1"

## 2. Create a loader

`TrialHandler` reads the datastore metadata and resolves each zarr-store path. The cells below use `lookup_trials(...)` to select only the rows needed for each load.

In [ ]:
loader = TrialHandler(DATA_ROOT, version=VERSION)

## 3. Select a small trial table

`lookup_trials(...)` accepts metadata columns as keyword filters. The returned table can be passed directly to `get_data(...)`.

This demo chooses a very small subset from one subject and one run, then loads only four named channels and a short time window. If the matching zarr store is not local, `lookup_trials` will ask whether to download it from S3.

In [ ]:
SUBJECT = 1
SESSION = 1
RUN = 1
N_TRIALS = 8
# Keep the demo small: load only these channels by name.
# If these names are not present in your local store, inspect `all_channel_names` below and edit this list.
CHANNELS = ["A1", "A2", "A3", "A4"]
TMIN = -0.1
TMAX = 0.8

trials = loader.lookup_trials(subject=SUBJECT, session=SESSION, run=RUN).head(N_TRIALS)

display_cols = [
    "subject",
    "session",
    "run",
    "epoch",
    "condition",
    "trial_type",
    "shared",
    "stim_file",
]
trials[[col for col in display_cols if col in trials.columns]]

## 4. Load single-trial epochs into memory

`get_data(...)` returns a dictionary with:

- `data`: NumPy array shaped `(n_trials, n_channels, n_samples)`.
- `metadata`: DataFrame aligned to the first axis of `data`.

Use `channels`, `tmin`, and `tmax` to keep the load small. Channels can be integer indices or channel names; channel names are usually clearer in analysis notebooks.

In [ ]:
# Peek at the first selected store so we can verify channel names before loading.
store = loader.store_cache.get(trials.loc[0, "path"])
if store is None:
    import zarr
    store = zarr.open(trials.loc[0, "path"], mode="r")

all_channel_names = store.attrs["info"]["ch_names"]
missing_channels = [ch for ch in CHANNELS if ch not in all_channel_names]

if missing_channels:
    raise ValueError(
        f"Channels not found: {missing_channels}. "
        f"First available names are: {all_channel_names[:20]}"
    )

CHANNELS

In [ ]:
result = loader.get_data(
    trials,
    channels=CHANNELS,
    tmin=TMIN,
    tmax=TMAX,
)

epochs = result["data"]
metadata = result["metadata"]

print("data shape:", epochs.shape)
print("dtype:", epochs.dtype)
metadata.head()

The zarr store keeps useful recording info in its attributes. Here we use it to recover channel names and the cropped time vector for plotting.

In [ ]:
store = loader.store_cache[trials.loc[0, "path"]]
all_times = np.asarray(store.attrs["times"])

tmin_idx = np.abs(all_times - TMIN).argmin()
tmax_idx = np.abs(all_times - TMAX).argmin()
times = all_times[tmin_idx:tmax_idx]
channel_names = CHANNELS

print(channel_names)
print(times[0], times[-1], len(times))

## 5. Plot a trial and a simple average

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times, epochs[0].T)
ax.axvline(0, color="black", linewidth=1, alpha=0.6)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Single trial, first 4 channels")
ax.legend(channel_names, loc="upper right", frameon=False)
plt.show()

In [ ]:
mean_epoch = epochs.mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times, mean_epoch.T)
ax.axvline(0, color="black", linewidth=1, alpha=0.6)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Average across selected trials")
ax.legend(channel_names, loc="upper right", frameon=False)
plt.show()

## 6. Trial averaging with `average_by`

`average_by` performs the grouping while loading. The returned metadata has one row per group and remains aligned to `data`.

For this example, we first find a couple of conditions with repeated trials for the selected subject. Then `get_data(..., average_by="condition")` returns one averaged epoch per condition.

In [ ]:
N_AVERAGE_CONDITIONS = 2
TRIALS_PER_CONDITION = 2

condition_counts = loader.metadata.loc[loader.metadata["subject"] == SUBJECT, "condition"].value_counts()
average_conditions = condition_counts[condition_counts >= TRIALS_PER_CONDITION].head(N_AVERAGE_CONDITIONS).index.tolist()
assert average_conditions, "No repeated conditions found for this subject; try a different SUBJECT."

average_trials = (
    loader.lookup_trials(subject=SUBJECT, condition=average_conditions)
    .sort_values(["condition", "session", "run", "epoch"])
    .groupby("condition", as_index=False)
    .head(TRIALS_PER_CONDITION)
    .reset_index(drop=True)
)

average_trials[[col for col in display_cols if col in average_trials.columns]]

In [ ]:
avg_result = loader.get_data(
    average_trials,
    channels=CHANNELS,
    tmin=TMIN,
    tmax=TMAX,
    average_by="condition",
)

print("input trial rows:", len(average_trials))
print("averaged rows:", len(avg_result["metadata"]))
print("averaged data shape:", avg_result["data"].shape)
avg_result["metadata"].head()

## 7. Iterate in batches for larger selections

For larger analyses, use `iter_data(...)` so only one batch is in memory at a time.

In [ ]:
total_rows = 0

for batch_idx, batch in enumerate(loader.iter_data(
    trials,
    batch_size=4,
    channels=CHANNELS,
    tmin=TMIN,
    tmax=TMAX,
), start=1):
    total_rows += len(batch["metadata"])
    print(f"batch {batch_idx}:", batch["data"].shape, batch["metadata"].shape)

print("total rows loaded:", total_rows)

## Common loading patterns

```python
# Inline filtering: lookup_trials is called inside get_data.
loader.get_data(subject=1, session=1, run=1, channels=["A1", "A2"], tmin=0.0, tmax=0.5)

# Shared images only.
shared_trials = loader.lookup_trials(subject=1, shared=True)

# Specific NSD image conditions.
image_trials = loader.lookup_trials(subject=1, condition=[5, 2951])

# First presentation only, then average by image condition.
first_repeats = loader.lookup_trials(subject=1, trial_instance=1)
loader.get_data(first_repeats, average_by="condition")
```